# Desafio 3 — Titanic: Machine Learning from Disaster

**Grupo:** a definir  
**Integrantes:** Vitor, Vilela e Wagner

## Objetivo

Construir e documentar um modelo de classificação capaz de prever a sobrevivência dos passageiros do Titanic.

> Este notebook é uma estrutura inicial. Cada seção deve ser desenvolvida, revisada e explicada pelo grupo.


## Responsabilidades

- **Vitor:** estrutura, pipeline, Random Forest, integração, submissão e entrega.
- **Vilela:** EDA, tratamento, engenharia de atributos, Regressão Logística e comparação.
- **Wagner:** requisitos, interpretação, documentação e validação funcional.


## 1. Importação das bibliotecas


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Carregamento dos dados


In [ ]:
# Ajuste os caminhos caso o notebook seja executado no Google Colab.
DATA_DIR = Path("../data/raw")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        "Inclua train.csv e test.csv em data/raw/ ou ajuste DATA_DIR."
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Treino:", train_df.shape)
print("Teste:", test_df.shape)
display(train_df.head())


## 3. Validação inicial dos dados


In [ ]:
print("Colunas do treino:")
print(train_df.columns.tolist())

print("\nTipos:")
display(train_df.dtypes.to_frame("dtype"))

print("\nValores ausentes:")
missing = (
    train_df.isna()
    .sum()
    .to_frame("quantidade")
    .assign(percentual=lambda x: x["quantidade"] / len(train_df) * 100)
    .sort_values("percentual", ascending=False)
)
display(missing)

print("\nDuplicados:", train_df.duplicated().sum())
display(train_df.describe(include="all").T)


## 4. Análise exploratória — Vilela

Para cada visualização:

1. criar o gráfico;
2. incluir título e eixos;
3. registrar uma interpretação objetiva;
4. enviar a interpretação para revisão do Wagner.


In [ ]:
# Distribuição da variável-alvo
survival_counts = train_df["Survived"].value_counts().sort_index()
survival_counts.plot(kind="bar")
plt.title("Distribuição da sobrevivência")
plt.xlabel("Sobreviveu")
plt.ylabel("Quantidade")
plt.xticks([0, 1], ["Não", "Sim"], rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Sobrevivência por sexo
survival_by_sex = pd.crosstab(
    train_df["Sex"], train_df["Survived"], normalize="index"
)
display(survival_by_sex)

survival_by_sex[1].plot(kind="bar")
plt.title("Taxa de sobrevivência por sexo")
plt.xlabel("Sexo")
plt.ylabel("Taxa de sobrevivência")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Sobrevivência por classe
survival_by_class = train_df.groupby("Pclass")["Survived"].mean()
display(survival_by_class)

survival_by_class.plot(kind="bar")
plt.title("Taxa de sobrevivência por classe")
plt.xlabel("Classe")
plt.ylabel("Taxa de sobrevivência")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Interpretação da EDA — Wagner

[Registrar aqui as interpretações dos gráficos sem confundir associação com causalidade.]


## 5. Engenharia de atributos — Vilela


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = (result["FamilySize"] == 1).astype(int)

    result["Title"] = (
        result["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
        .replace(
            {
                "Mlle": "Miss",
                "Ms": "Miss",
                "Mme": "Mrs",
            }
        )
    )

    rare_titles = ~result["Title"].isin(["Mr", "Mrs", "Miss", "Master"])
    result.loc[rare_titles, "Title"] = "Rare"

    return result


train_model_df = add_features(train_df)
test_model_df = add_features(test_df)

display(train_model_df[["Name", "Title", "FamilySize", "IsAlone"]].head())


## 6. Seleção das variáveis e separação treino/validação

A lista abaixo é inicial. Vitor e Vilela devem revisar juntos.


In [ ]:
TARGET = "Survived"

numeric_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "FamilySize",
    "IsAlone",
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
]

selected_features = numeric_features + categorical_features

X = train_model_df[selected_features]
y = train_model_df[TARGET]
X_test = test_model_df[selected_features]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_valid.shape)


## 7. Pipeline de pré-processamento — Vitor


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)


## 8. Modelos


In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=6,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

models = {
    "Regressão Logística": logistic_model,
    "Random Forest": random_forest_model,
}


## 9. Avaliação e comparação


In [ ]:
def evaluate_model(name: str, model: Pipeline) -> dict:
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)

    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="accuracy",
    )

    print(f"\n{name}")
    print("Matriz de confusão:")
    print(confusion_matrix(y_valid, predictions))
    print("\nRelatório:")
    print(classification_report(y_valid, predictions))

    return {
        "Modelo": name,
        "Accuracy": accuracy_score(y_valid, predictions),
        "Precision": precision_score(y_valid, predictions, zero_division=0),
        "Recall": recall_score(y_valid, predictions, zero_division=0),
        "F1": f1_score(y_valid, predictions, zero_division=0),
        "CV média": cv_scores.mean(),
        "CV desvio": cv_scores.std(),
    }


results = []
for model_name, model in models.items():
    results.append(evaluate_model(model_name, model))

results_df = pd.DataFrame(results).sort_values("CV média", ascending=False)
display(results_df)


## 10. Escolha do modelo final — Vitor e Vilela

Registrar:

- modelo selecionado;
- justificativa;
- desempenho;
- estabilidade;
- interpretabilidade;
- possível overfitting.


In [ ]:
best_model_name = results_df.iloc[0]["Modelo"]
best_model = models[best_model_name]

print("Modelo selecionado:", best_model_name)


## 11. Treinamento final e geração da submissão — Vitor


In [ ]:
best_model.fit(X, y)
test_predictions = best_model.predict(X_test).astype(int)

submission = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions,
    }
)

OUTPUT_DIR = Path("../outputs/submission")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

submission.to_csv(SUBMISSION_PATH, index=False)

print("Arquivo gerado:", SUBMISSION_PATH)
display(submission.head())
print(submission.shape)


## 12. Resultado da submissão — Wagner

- **Data:**
- **Modelo:**
- **Pontuação:**
- **Screenshot:**
- **Observações:**


## 13. Conclusão

[Wagner redige. Vitor e Vilela revisam tecnicamente.]

Incluir:

- principais padrões observados;
- comparação entre os modelos;
- modelo escolhido;
- limitações;
- possíveis evoluções.
